<h1 align="center">Practice 4. APIs & Static Web Scraping: REST API requests, JSON extraction + 
BeautifulSoup static HTML parsing.</h1>

### Topic 

Books

### Imports

In [1]:
import re
from time import sleep
import requests as req
import pandas as pd
from bs4 import BeautifulSoup

## Task 1. Choose data sources 

### Selected data sources:

1. API: [Gutendex](https://gutendex.com/)
2. Static Webpage: [Books to Scrape](https://books.toscrape.com/index.html)

## Task 2. API data collection

In [2]:
API_BASE_URL = "https://gutendex.com/"

In [3]:
# list of books
response = req.get(url=f"{API_BASE_URL}books")
status = response.status_code
books_data = response.json()
print(f"{status=} {type(books_data)}")
print(f"keys: {books_data.keys()}")

status=200 <class 'dict'>
keys: dict_keys(['count', 'next', 'previous', 'results'])


In [4]:
# single book
books_data["results"][0].keys()

dict_keys(['id', 'title', 'authors', 'summaries', 'editors', 'translators', 'subjects', 'bookshelves', 'languages', 'copyright', 'media_type', 'formats', 'download_count'])

### Exctracting 1000 books

Each page contains 32 book records, 32 get requests needed

In [ ]:
books_records_list = []
for page in range(1, 33):
    params = {
        "page": page,
        "sort": "ascending"
    }
    response = req.get(url=f"{API_BASE_URL}/books", params=params)
    print(f"{page=} status: {response.status_code}")
    data = response.json()
    books_records_list.extend(data["results"])
    sleep(0.1)
print(f"First record example title: {books_records_list[0]["title"]}")

page=1 status: 200
page=2 status: 200
page=3 status: 200
page=4 status: 200
page=5 status: 200
page=6 status: 200
page=7 status: 200
page=8 status: 200
page=9 status: 200
page=10 status: 200
page=11 status: 200
page=12 status: 200
page=13 status: 200
page=14 status: 200
page=15 status: 200
page=16 status: 200
page=17 status: 200
page=18 status: 200
page=19 status: 200


In [ ]:
books_api_df = pd.DataFrame(
    books_records_list
)
books_api_df = books_api_df[["id", "title", "authors", "summaries", "languages", "copyright", "download_count"]]

## Task 3. JSON processing

In [ ]:
books_api_df.head()

In [ ]:
books_api_df.info()

In [ ]:
def parse_authors(authors: list[dict[str, str]]) -> str:
    if not authors: return pd.NA
    return "|".join(author["name"] for author in authors)
    
def parse_summaries(summaries: list[str]) -> str:
    if not summaries: return pd.NA
    return ",".join(summaries)

def parse_languages(languages: list[str]) -> str:
    if not languages: return pd.NA
    return ",".join(languages)

In [ ]:
books_api_df["authors"] = books_api_df["authors"].apply(parse_authors).astype(str)
books_api_df["summaries"] = books_api_df["summaries"].apply(parse_summaries).astype(str)
books_api_df["languages"] = books_api_df["languages"].apply(parse_languages).astype(str)

In [ ]:
books_api_df.head()

In [ ]:
books_api_df.dtypes

In [ ]:
books_api_df.to_csv("./lab4/books_api_dataset.csv", index=False)

## Task 4. Web scraping

In [ ]:
STATIC_WEBPAGE_URL = "https://books.toscrape.com"

In [ ]:
response = req.get(url=STATIC_WEBPAGE_URL)
print(f"Status: {response.status_code}")
html_text = response.text

In [ ]:
soup = BeautifulSoup(html_text)

In [ ]:
book_details_links = []
for card in soup.body.find_all(class_="product_pod"):
    book_details_links.append(card.find(class_="thumbnail").parent.get("href"))

In [ ]:
book_details_links

In [ ]:
scraping_books_records_list = []
for link in book_details_links:
    response = req.get(url=f"{STATIC_WEBPAGE_URL}/{link}")
    print(f"{link.split("/")[-2]} status: {response.status_code}")
    soup = BeautifulSoup(response.text)
    product_main_info = soup.find(class_="product_main")
    product_table_info = soup.find("table")
    scraping_books_records_list.append(
        {
            "title": product_main_info.h1.string,
            "price": product_main_info.h1.next_sibling.next_sibling.string,
            "status": product_main_info.find(class_="instock").contents[-1],
            "UPC": product_table_info.find("tr").td.string
        }
    )
    sleep(0.1)
scraping_books_records_list

In [ ]:
for book in scraping_books_records_list:
    book["price"] = re.sub(r"[^0-9.]", "", book["price"])
    book["status"] = book["status"].strip()
scraping_books_records_list

In [ ]:
books_scraped_df = pd.DataFrame(
    scraping_books_records_list
)

In [ ]:
books_scraped_df.head()

In [ ]:
books_scraped_df.info()

In [ ]:
books_scraped_df["price"] = books_scraped_df["price"].astype("float64")

In [ ]:
books_scraped_df.dtypes

In [ ]:
books_scraped_df.to_csv("./lab4/books_scraped_dataset.csv", index=False)

## Task 5. Data review

### API Dataset

In [ ]:
books_api_df.head()

In [ ]:
books_api_df.shape

In [ ]:
books_api_df.info()

#### Finding 1

In [ ]:
books_api_df[books_api_df["authors"].isna()]

4 books have unknown authors, though missing values considered to be changed or deleted, in this case they are allowed as some books doesn't have authors at all (combined work of many story tellers as an example).

#### Finding 2

In [ ]:
books_api_df["languages"].value_counts()